In [1]:
import sys
sys.path.append("..")

import torch
from src.training.loss import LaplaceLoss, laplace_metric
from src.utils.seed import set_seed

set_seed(42)

mu    = torch.tensor([2500.0, 3000.0, 2200.0])
sigma = torch.tensor([150.0,  200.0,  80.0])
fvc   = torch.tensor([2400.0, 2950.0, 2300.0])

loss = LaplaceLoss()(mu, sigma, fvc)
print(f"LaplaceLoss value : {loss.item():.4f}")
print(f"Expected          : positive number around 7-8")

# Edge case: sigma below minimum should be clipped to 70
sigma_low = torch.tensor([10.0, 20.0, 30.0])
loss_clipped = LaplaceLoss()(mu, sigma_low, fvc)
print(f"\nWith sigma clipped: {loss_clipped.item():.4f}")
print("Both should work without errors — PASS")

LaplaceLoss value : 6.2649
Expected          : positive number around 7-8

With sigma clipped: 6.2787
Both should work without errors — PASS


In [2]:
import numpy as np
from src.training.loss import laplace_metric

score = laplace_metric(
    mu       = np.array([2500., 3000., 2200.]),
    sigma    = np.array([150.,  200.,  80.]),
    fvc_true = np.array([2400., 2950., 2300.])
)
print(f"Laplace metric : {score:.4f}")
print(f"Should be      : negative number (higher = better, max = 0)")
print(f"PASS" if score < 0 else "FAIL — something wrong")

Laplace metric : -6.2649
Should be      : negative number (higher = better, max = 0)
PASS


In [3]:
import pandas as pd
from src.models.tabular import prepare_features

dummy = pd.DataFrame({
    "Patient":       ["P001","P001","P002","P002"],
    "Weeks":         [0, 20, 5, 30],
    "FVC":           [2500, 2400, 3000, 2900],
    "Percent":       [70.0, 68.0, 80.0, 78.0],
    "Age":           [65, 65, 70, 70],
    "Sex":           ["Male","Male","Female","Female"],
    "SmokingStatus": ["Ex-smoker","Ex-smoker","Never smoked","Never smoked"]
})

out = prepare_features(dummy)
print(out[["Patient","Weeks","WeekOffset","FVCpct_norm","Sex","SmokingStatus"]])

# Checks
assert out["WeekOffset"].tolist() == [0, 20, 0, 25], "WeekOffset wrong"
assert out["FVCpct_norm"].tolist() == [0.7, 0.68, 0.8, 0.78], "FVCpct wrong"
assert out["Sex"].dtype in [int, "int64", "int32"], "Sex not encoded"
print("\nAll assertions passed — PASS")

  Patient  Weeks  WeekOffset  FVCpct_norm  Sex  SmokingStatus
0    P001      0           0         0.70    1              0
1    P001     20          20         0.68    1              0
2    P002      5           0         0.80    0              1
3    P002     30          25         0.78    0              1

All assertions passed — PASS


In [4]:
print("=" * 40)
print("Phase 1 local checks complete")
print("=" * 40)
print()
print("Loss function   : PASS")
print("Numpy metric    : PASS")
print("Feature prep    : PASS")
print()
print("Ready to push and run on Kaggle.")

Phase 1 local checks complete

Loss function   : PASS
Numpy metric    : PASS
Feature prep    : PASS

Ready to push and run on Kaggle.
